# 철도 시계열 로더

### 폴더 구조
```
GTFS_CT/
├── 철도망/          ← AF0302 교차점(역), AF0022 중심선
│   ├── 2016/
│   ├── 2017/
│   └── ...2024/
└── 철도/            ← Rail_node, Rail_route, Rail_route_station, Rail_time_table
    ├── 2022/
    └── ...2025/
```

### 연도별 스키마 변경 이력
| 연도 | AF0302 변경 | AF0022 변경 |
|------|------------|------------|
| 2016 | FACILITY_ID 있음 | 최초 구축 |
| 2017 | FACILITY_ID → RAILTRANSF (환승유형 추가) | - |
| 2018+ | 환승역 분할 적용 | - |

In [1]:
import re
from pathlib import Path
import pandas as pd
import geopandas as gpd

In [2]:
BASE_DIR   = Path(r"C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT")
RAIL_DIR   = BASE_DIR / "철도망"   # AF0302, AF0022
TRANSIT_DIR = BASE_DIR / "철도"    # Rail_node, Rail_route, Rail_route_station, Rail_time_table

# 아산시 + 천안 인접 권역 (WGS84)
# 철도는 전국 망이라 역 수가 적으므로 충청권 전체로 넓게 잡아도 무방
ASAN_BOUNDS_WGS84 = (126.8, 36.6, 127.1, 36.9)   # 아산시 한정
CHUNGNAM_BOUNDS   = (126.0, 36.0, 128.0, 37.5)    # 충남 전체 (선택)

In [3]:
# ── shp 필드명 → 풀네임 정규화 맵 ──────────────────────────────
# AF0302 (철도 교차점/역)
NODE_RENAME = {
    "RAILNODE_I": "RAILNODE_ID",
    "RAILNODE_T": "RAILNODE_TYPE",
    "STATION_NA": "STATION_NAME",
    "STATION_N2": "STATION_NAME_SUB",
    "RAILTRANSF": "RAILTRANSFER_TYPE",
    "OPENNESS_S": "OPENNESS_STATUS",
    "MANAGING_A": "MANAGING_AGENCY",
    "DISTRICT_I": "DISTRICT_ID",
    "SERVICE_TY": "SERVICE_TYPE",
    "RN_HISTORY": "RN_HISTORY",
    "MODIFY_CHE": "MODIFY_CHECK",
    "MODIFY_DAT": "MODIFY_DATE",
    "SURVEY_DAT": "SURVEY_DATE",
    # 2016 구버전 필드명
    "RAILNODE_S": "RAILNODE_TYPE",   # 일부 연도 shp 필드명 차이
}

# AF0022 (철도 중심선)
LINK_RENAME = {
    "RAILLINK_I": "RAILLINK_ID",
    "FROM_RAILN": "FROM_RAILNODE_ID",
    "TO_RAILNOD": "TO_RAILNODE_ID",
    "RAILLINE_N": "RAILLINE_NAME1",
    "RAILLINEN2": "RAILLINE_NAME2",
    "RAILLINEN3": "RAILLINE_NAME3",
    "RAILLINE_I": "RAILLINE_ID1",
    "RAILLINEI2": "RAILLINE_ID2",
    "RAILLINEI3": "RAILLINE_ID3",
    "RAIL_TYPE":  "RAIL_TYPE",
    "MANAGING_A": "MANAGING_AGENCY",
    "ELECTRONIC": "ELECTRONICRAIL",
    "RAILWAY_RA": "RAILWAY_RANK",
    "OPENNESS_S": "OPENNESS_STATUS",
    "DISTIRCT_I": "DISTRICT_ID",    # 오타 필드명도 처리
    "DISTRICT_I": "DISTRICT_ID",
    "RL_HISTORY": "RL_HISTORY",
    "MODIFY_CHE": "MODIFY_CHECK",
    "MODIFY_DAT": "MODIFY_DATE",
    "SURVEY_DAT": "SURVEY_DATE",
}

# Rail_node (대중교통 철도 노드)
RNODE_RENAME = {
    "MODIFY_CHE": "MODIFY_CHECK",
    "MODIFY_DAT": "MODIFY_DATE",
    "SURVEY_DAT": "SURVEY_DATE",
    "DISTRICT_I": "DISTRICT_ID",
}

# Rail_route (대중교통 철도 노선)
RROUTE_RENAME = {
    "VEHICLE_TY": "VEHICLE_TYPE",
    "TT_OP_COUN": "TT_OP_COUNT",
    "MODIFY_CHE": "MODIFY_CHECK",
    "MODIFY_DAT": "MODIFY_DATE",
    "SURVEY_DAT": "SURVEY_DATE",
}

In [4]:
def find_shp(folder: Path, keyword: str) -> Path | None:
    """폴더 내 keyword 포함 .shp 반환 (재귀)"""
    for shp in folder.rglob("*.shp"):
        if keyword.lower() in shp.stem.lower():
            return shp
    shps = list(folder.rglob("*.shp"))
    return shps[0] if shps else None


def find_dbf(folder: Path, keyword: str) -> Path | None:
    """폴더 내 keyword 포함 .dbf 반환 (재귀)"""
    for dbf in folder.rglob("*.dbf"):
        if keyword.lower() in dbf.stem.lower():
            return dbf
    dbfs = list(folder.rglob("*.dbf"))
    return dbfs[0] if dbfs else None


def find_type_dir(year_dir: Path, keywords: list[str]) -> Path | None:
    """연도 폴더 하위에서 keywords 매칭 폴더 탐색 (도로망 로더와 동일 패턴)"""
    for sub in sorted(year_dir.iterdir()):
        if not sub.is_dir():
            continue
        name = sub.name.lower().replace(" ", "").replace(".", "")
        if any(k in name for k in keywords):
            return sub
        for subsub in sub.iterdir():
            if subsub.is_dir():
                n2 = subsub.name.lower().replace(" ", "").replace(".", "")
                if any(k in n2 for k in keywords):
                    return subsub
    return year_dir


def load_shp(path: Path, rename_map: dict, year: int) -> gpd.GeoDataFrame | None:
    if path is None or not path.exists():
        return None
    try:
        gdf = gpd.read_file(path, engine="pyogrio", encoding='cp949')
        gdf.columns = [c.upper() for c in gdf.columns]
        gdf = gdf.rename(columns={k.upper(): v.upper() for k, v in rename_map.items()})
        
        # GEOMETRY 컬럼을 geometry로 설정
        if 'GEOMETRY' in gdf.columns:
            gdf = gdf.set_geometry('GEOMETRY')
        
        gdf["data_year"] = year
        return gdf
    except Exception as e:
        print(f"  [ERR] {path.name} ({year}): {e}")
        return None


def load_dbf(path: Path, rename_map: dict, year: int) -> pd.DataFrame | None:
    """DBF 전용 로더 (Rail_time_table 등 geometry 없는 파일)"""
    if path is None or not path.exists():
        return None
    try:
        # geopandas로 dbf 읽기
        df = gpd.read_file(path, engine="pyogrio", encoding='cp949')
        df = pd.DataFrame(df.drop(columns=["geometry"], errors="ignore"))
        df.columns = [c.upper() for c in df.columns]
        df = df.rename(columns={k.upper(): v.upper() for k, v in rename_map.items()})
        df["data_year"] = year
        return df
    except Exception as e:
        print(f"  [ERR] {path.name} ({year}): {e}")
        return None


def filter_bbox(gdf: gpd.GeoDataFrame, bounds: tuple) -> gpd.GeoDataFrame:
    """좌표계 변환 후 bounding box 클립"""
    if gdf.crs is None:
        print("  [WARN] CRS 없음 - EPSG:5179 가정")
        gdf = gdf.set_crs(epsg=5179)
    wgs = gdf.to_crs(epsg=4326)
    minx, miny, maxx, maxy = bounds
    return wgs.cx[minx:maxx, miny:maxy].copy()

In [5]:
# 폴더 구조 확인
for base, label in [(RAIL_DIR, "철도망"), (TRANSIT_DIR, "철도")]:
    if not base.exists():
        print(f"[없음] {base}")
        continue
    print(f"\n[{label}] {base}")
    for year_dir in sorted(base.iterdir()):
        if not year_dir.is_dir():
            continue
        subs = [s.name for s in sorted(year_dir.iterdir()) if s.is_dir()]
        shps = list(year_dir.rglob("*.shp"))
        dbfs = list(year_dir.rglob("*.dbf"))
        print(f"  {year_dir.name}/  하위: {subs}  shp:{len(shps)}  dbf:{len(dbfs)}")


[철도망] C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT\철도망
  2016/  하위: ['[01]철도교차점', '[02]철도중심선']  shp:2  dbf:2
  2017/  하위: ['[01]철도교차점', '[02]철도중심선']  shp:2  dbf:2
  2018/  하위: ['01. 철도교차점', '02. 철도중심선']  shp:2  dbf:2
  2019/  하위: ['01. 철도교차점', '02. 철도중심선']  shp:2  dbf:2
  2020/  하위: ['2020-TM-GR-MR-AML 철도망(2019년 기준)']  shp:2  dbf:2
  2021/  하위: ['2021-TM-GR-MR-AML 철도망(2020년 기준)']  shp:2  dbf:2
  2022/  하위: ['2022-TM-GR-MR-AML 철도망(2021년 기준)']  shp:2  dbf:2
  2023/  하위: ['2023-TM-GR-MR-AML 철도망(2022년 기준)']  shp:2  dbf:2
  2024/  하위: ['01. 철도교차점', '02. 철도중심선']  shp:2  dbf:2
  _csv/  하위: []  shp:0  dbf:0

[철도] C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT\철도
  2016/  하위: ['[1] 노드', '[2] 노선', '[3] 노선 정류장리스트', '[4] 시각표']  shp:4  dbf:4
  2017/  하위: ['[1] 노드', '[2] 노선', '[3] 노선 정류장리스트', '[4] 시각표']  shp:4  dbf:4
  2018/  하위: ['[1] 노드', '[2] 노선', '[3] 노선 정류장리스트', '[4] 시각표']  shp:4  dbf:4
  2019/  하위: ['[1] 노드', '[2] 노선', '[3] 노선 정류장리스트

---
## 1. 철도망 GIS DB (AF0302 교차점 + AF0022 중심선)

In [6]:
rail_nodes, rail_links = [], []

for year_dir in sorted(RAIL_DIR.iterdir()):
    if not year_dir.is_dir() or not year_dir.name.isdigit():
        continue
    year = int(year_dir.name)

    # AF0302 교차점(역) 폴더
    node_dir = find_type_dir(year_dir, ["교차점", "node", "af0302"])
    # AF0022 중심선 폴더
    link_dir = find_type_dir(year_dir, ["중심선", "link", "af0022"])

    node_shp = find_shp(node_dir, "af0302") or find_shp(node_dir, "node")
    link_shp = find_shp(link_dir, "af0022") or find_shp(link_dir, "link")

    gdf_node = load_shp(node_shp, NODE_RENAME, year)
    gdf_link = load_shp(link_shp, LINK_RENAME, year)

    if gdf_node is not None:
        # GEOMETRY 컬럼을 geometry로 설정
        if 'GEOMETRY' in gdf_node.columns:
            gdf_node = gdf_node.set_geometry('GEOMETRY')

        rail_nodes.append(gdf_node)
        print(f"  [OK] {year} AF0302  {len(gdf_node):>5,}행  crs={gdf_node.crs}")

    else:
        print(f"  [--] {year} AF0302  shp 없음")

    if gdf_link is not None:
        rail_links.append(gdf_link)
        print(f"  [OK] {year} AF0022  {len(gdf_link):>5,}행")
    else:
        print(f"  [--] {year} AF0022  shp 없음")

railnet = {
    "node": pd.concat(rail_nodes, ignore_index=True) if rail_nodes else None,
    "link": pd.concat(rail_links, ignore_index=True) if rail_links else None,
}
print(f"\n로드 완료: {[k for k,v in railnet.items() if v is not None]}")

  [OK] 2016 AF0302  1,273행  crs=PROJCS["Korea 2000 Katech(TM128)",GEOGCS["ITRF2000",DATUM["International_Terrestrial_Reference_Frame_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6656"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",128],PARAMETER["scale_factor",0.9999],PARAMETER["false_easting",400000],PARAMETER["false_northing",600000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
  [OK] 2016 AF0022  1,382행
  [OK] 2017 AF0302  1,324행  crs=PROJCS["Korea 2000 Katech(TM128)",GEOGCS["ITRF2000",DATUM["International_Terrestrial_Reference_Frame_2000",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6656"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",128]

---
## 2. 대중교통(철도) GIS DB (Rail_node / route / station / timetable)

In [7]:
t_nodes, t_routes, t_stations, t_timetables = [], [], [], []

for year_dir in sorted(TRANSIT_DIR.iterdir()):
    if not year_dir.is_dir():
        continue
    # 연월 폴더명에서 연도 추출 (예: 2022, 202203 등)
    m = re.search(r"(\d{4})", year_dir.name)
    if not m:
        continue
    year = int(m.group(1))

    node_shp    = find_shp(year_dir, "rail_node")    or find_shp(year_dir, "railnode")
    route_shp   = find_shp(year_dir, "rail_route")   or find_shp(year_dir, "railroute")
    station_shp = find_shp(year_dir, "rail_route_station") or find_shp(year_dir, "routestation")
    # Rail_time_table은 보통 DBF (geometry 없음)
    timetable_dbf = find_dbf(year_dir, "time_table") or find_dbf(year_dir, "timetable")

    gdf_node    = load_shp(node_shp,    RNODE_RENAME,  year)
    gdf_route   = load_shp(route_shp,   RROUTE_RENAME, year)
    gdf_station = load_shp(station_shp, {},            year)
    df_timetable = load_dbf(timetable_dbf, {}, year)

    if gdf_node    is not None: t_nodes.append(gdf_node);           print(f"  [OK] {year} Rail_node          {len(gdf_node):>6,}행")
    if gdf_route   is not None: t_routes.append(gdf_route);         print(f"  [OK] {year} Rail_route         {len(gdf_route):>6,}행")
    if gdf_station is not None: t_stations.append(gdf_station);     print(f"  [OK] {year} Rail_route_station {len(gdf_station):>6,}행")
    if df_timetable is not None: t_timetables.append(df_timetable); print(f"  [OK] {year} Rail_time_table    {len(df_timetable):>6,}행")

transit = {
    "rail_node":           pd.concat(t_nodes,      ignore_index=True) if t_nodes      else None,
    "rail_route":          pd.concat(t_routes,     ignore_index=True) if t_routes     else None,
    "rail_route_station":  pd.concat(t_stations,   ignore_index=True) if t_stations   else None,
    "rail_time_table":     pd.concat(t_timetables, ignore_index=True) if t_timetables else None,
}
print(f"\n로드 완료: {[k for k,v in transit.items() if v is not None]}")

  [OK] 2016 Rail_node           1,249행
  [OK] 2016 Rail_route          1,391행
  [OK] 2016 Rail_route_station 27,235행
  [OK] 2016 Rail_time_table    23,458행
  [OK] 2017 Rail_node           1,300행
  [OK] 2017 Rail_route          1,519행
  [OK] 2017 Rail_route_station 28,702행
  [OK] 2017 Rail_time_table    25,219행
  [OK] 2018 Rail_node           1,523행
  [OK] 2018 Rail_route          1,012행
  [OK] 2018 Rail_route_station 16,396행
  [OK] 2018 Rail_time_table    26,617행
  [OK] 2019 Rail_node           1,551행
  [OK] 2019 Rail_route          1,080행
  [OK] 2019 Rail_route_station 18,662행
  [OK] 2019 Rail_time_table    12,888행
  [OK] 2020 Rail_node           1,565행
  [OK] 2020 Rail_route          1,601행
  [OK] 2020 Rail_route_station 30,354행
  [OK] 2020 Rail_time_table    26,247행
  [OK] 2021 Rail_node           1,586행
  [OK] 2021 Rail_route          1,652행
  [OK] 2021 Rail_route_station 31,796행
  [OK] 2021 Rail_time_table    25,734행
  [OK] 2022 Rail_node           1,576행
  [OK] 2022 Rail_route   

C:\Users\HP\IdeaProjects\sundo\asan\sundo_asan\.venv\Lib\site-packages\geopandas\array.py:1770: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as Korean 1985 Katech(TM128) (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))
C:\Users\HP\IdeaProjects\sundo\asan\sundo_asan\.venv\Lib\site-packages\geopandas\array.py:1770: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as Korean 1985 Katech(TM128) (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))
C:\Users\HP\IdeaProjects\sundo\asan\sundo_asan\.venv\Lib\site-packages\geopandas\array.py:1770: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as Korean 1985 Katech(TM128) (the single non-null crs provided).
  return GeometryArray(data, crs=_get_common_crs(to_concat))


---
## 3. 아산시 필터링

In [8]:
asan_railnet  = {}
asan_transit  = {}

# 철도망: geometry 있는 것만 bbox 필터
for name, gdf in railnet.items():
    if gdf is None:
        continue
    if "geometry" in gdf.columns and gdf.geometry.notna().any():
        asan_railnet[name] = filter_bbox(gdf, ASAN_BOUNDS_WGS84)
        print(f"  railnet.{name}: 전국 {len(gdf):,}행 → 아산시 {len(asan_railnet[name]):,}행")
    else:
        asan_railnet[name] = gdf

# 대중교통 Rail_node 기준으로 아산 역 ID 추출 → 나머지 테이블 필터
for name, gdf in transit.items():
    if gdf is None:
        continue

    if name == "rail_node" and "geometry" in gdf.columns and gdf.geometry.notna().any():
        asan_transit[name] = filter_bbox(gdf, ASAN_BOUNDS_WGS84)
        print(f"  transit.{name}: 전국 {len(gdf):,}행 → 아산시 {len(asan_transit[name]):,}행")

    elif name in ("rail_route_station", "rail_time_table") and "rail_node" in asan_transit:
        # 아산 역 NODE_ID 기준 필터
        asan_node_ids = set(asan_transit["rail_node"]["NODE_ID"].dropna())
        filtered = gdf[gdf["NODE_ID"].isin(asan_node_ids)]
        asan_transit[name] = filtered
        print(f"  transit.{name}: 전국 {len(gdf):,}행 → 아산 역 기준 {len(filtered):,}행")

    elif name == "rail_route" and "rail_route_station" in asan_transit:
        # 아산 경유 노선 ROUTE_ID 기준 필터
        asan_route_ids = set(asan_transit["rail_route_station"]["ROUTE_ID"].dropna())
        filtered = gdf[gdf["ROUTE_ID"].isin(asan_route_ids)]
        asan_transit[name] = filtered
        print(f"  transit.{name}: 전국 {len(gdf):,}행 → 아산 경유 {len(filtered):,}행")

    else:
        asan_transit[name] = gdf

  transit.rail_route_station: 전국 250,039행 → 아산 역 기준 250,039행
  transit.rail_time_table: 전국 217,855행 → 아산 역 기준 217,855행


In [9]:
# 아산시 철도역 목록
if "rail_node" in asan_transit and asan_transit["rail_node"] is not None:
    cols = [c for c in ["data_year", "NODE_ID", "NODE_NAME", "NODE_TYPE", "DISTRICT_ID"] if c in asan_transit["rail_node"].columns]
    print(asan_transit["rail_node"][cols].drop_duplicates(subset=["NODE_NAME"]).to_string(index=False))

 data_year      NODE_ID        NODE_NAME NODE_TYPE DISTRICT_ID
      2016 RN_38_000060               통해     RN017       38115
      2016 RN_32_000070              서원주     RN017       32020
      2016 RN_31_000077               보정     RN013       31192
      2016 RN_31_000021               구성     RN013       31192
      2016 RN_31_000117               신갈     RN013       31192
      2016 RN_11_000201           양재시민의숲     RN013       11220
      2016 RN_11_000261            청계산입구     RN013       11220
      2016 RN_31_000194         판교(신분당선)     RN013       31023
      2016 RN_21_000010               공항     RN016       21120
      2016 RN_21_000084          서부산유통지구     RN016       21120
      2016 RN_21_000078        사상(서부터미널)     RN014       21120
      2016 RN_21_000012           괘법르네시떼     RN016       21120
      2016 RN_21_000037               덕두     RN016       21120
      2016 RN_21_000048               등구     RN016       21120
      2016 RN_21_000035               대저     RN014     

In [10]:
# 아산 경유 노선 목록
if "rail_route" in asan_transit and asan_transit["rail_route"] is not None:
    cols = [c for c in ["data_year", "ROUTE_ID", "R_GROUP", "ROUTE_NAME", "ROUTE_TYPE", "TT_OP_COUNT"] if c in asan_transit["rail_route"].columns]
    print(asan_transit["rail_route"][cols].sort_values(["data_year", "ROUTE_TYPE"]).to_string(index=False))

 data_year    ROUTE_ID               R_GROUP                              ROUTE_NAME ROUTE_TYPE  TT_OP_COUNT
      2016 RR_31_00001        고속철도-KTX경부선-하행                      KTX경부선-하행/1/광명-부산역      RR001            1
      2016 RR_31_00002        고속철도-KTX경부선-하행                      KTX경부선-하행/2/광명-부산역      RR001            3
      2016 RR_31_00003        고속철도-KTX경부선-하행                      KTX경부선-하행/3/광명-부산역      RR001            1
      2016 RR_31_00004        고속철도-KTX경부선-하행                      KTX경부선-하행/4/광명-부산역      RR001            1
      2016 RR_31_00005        고속철도-KTX경부선-하행                      KTX경부선-하행/5/광명-부산역      RR001            1
      2016 RR_31_00006        고속철도-KTX경부선-하행                      KTX경부선-하행/6/광명-부산역      RR001            1
      2016 RR_25_00007        고속철도-KTX경부선-상행                       KTX경부선-상행/1/대전-서울      RR001            3
      2016 RR_25_00008        고속철도-KTX경부선-상행                   KTX경부선-상행/2/대전-인천국제공항      RR001            1
      2016 RR_21_00

In [11]:
# 연도별 역수 + 노선수 요약
for name, df in {**asan_railnet, **asan_transit}.items():
    if df is None or "data_year" not in df.columns:
        continue
    summary = df.groupby("data_year").size().rename("건수")
    print(f"\n[{name}]")
    print(summary.to_string())


[node]
data_year
2016    1273
2017    1324
2018    1538
2019    1566
2020    1580
2021    1601
2022    1592
2023    1610
2024    1621

[link]
data_year
2016    1382
2017    1440
2018    1594
2019    1621
2020    1638
2021    1664
2022    1657
2023    1679
2024    1690

[rail_node]
data_year
2016    1249
2017    1300
2018    1523
2019    1551
2020    1565
2021    1586
2022    1576
2023    1596
2024    1605

[rail_route]
data_year
2016    1391
2017    1519
2018    1012
2019    1080
2020    1601
2021    1652
2022    1629
2023    1652
2024    1646

[rail_route_station]
data_year
2016    27235
2017    28702
2018    16396
2019    18662
2020    30354
2021    31796
2022    32287
2023    32573
2024    32034

[rail_time_table]
data_year
2016    23458
2017    25219
2018    26617
2019    12888
2020    26247
2021    25734
2022    25557
2023    26159
2024    25976


---
## 4. CSV 저장

In [12]:
def save_csv(df, path: Path):
    """geometry 제거 후 CP949 저장, 실패 시 utf-8-sig fallback"""
    if hasattr(df, "geometry") and "geometry" in df.columns:
        df = df.drop(columns=["geometry"])
    try:
        df.to_csv(path, index=False, encoding="cp949")
    except UnicodeEncodeError:
        path = path.with_stem(path.stem + "_utf8")
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"  [fallback utf8] {path.name}")
    print(f"  [저장] {path.name}  {df.shape}")


# 아산시 데이터 저장
SAVE_DIR = BASE_DIR / "철도망" / "_csv"
SAVE_DIR.mkdir(exist_ok=True)

for name, df in asan_railnet.items():
    if df is not None:
        save_csv(df, SAVE_DIR / f"railnet_{name}_asan.csv")

TRANSIT_SAVE_DIR = BASE_DIR / "철도" / "_csv"
TRANSIT_SAVE_DIR.mkdir(exist_ok=True)

for name, df in asan_transit.items():
    if df is not None:
        save_csv(df, TRANSIT_SAVE_DIR / f"{name}_asan.csv")

print(f"\n저장 경로:\n  {SAVE_DIR}\n  {TRANSIT_SAVE_DIR}")

  [저장] railnet_node_asan.csv  (13705, 31)
  [저장] railnet_link_asan.csv  (14365, 36)
  [저장] rail_node_asan.csv  (13551, 12)
  [저장] rail_route_asan.csv  (13182, 20)
  [저장] rail_route_station_asan.csv  (250039, 5)
  [저장] rail_time_table_asan.csv  (217855, 13)

저장 경로:
  C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT\철도망\_csv
  C:\Users\HP\Desktop\작업 폴더\01_work\아산시\01_데이터\원천데이터\교통약자_이동권_분석\GTFS\GTFS_CT\철도\_csv


In [13]:
# 전국 데이터 저장 (용량 주의: 중심선은 연도별 수만 행)
save_all = input("전국 데이터도 저장? (y/n): ").strip().lower() == "y"
if save_all:
    for name, df in {**railnet, **transit}.items():
        if df is not None:
            base = SAVE_DIR if name in ("node", "link") else TRANSIT_SAVE_DIR
            prefix = "railnet_" if name in ("node", "link") else ""
            save_csv(df, base / f"{prefix}{name}_all.csv")

  [저장] railnet_node_all.csv  (13705, 31)
  [저장] railnet_link_all.csv  (14365, 36)
  [저장] rail_node_all.csv  (13551, 12)
  [저장] rail_route_all.csv  (13182, 20)
  [저장] rail_route_station_all.csv  (250039, 5)
  [저장] rail_time_table_all.csv  (217855, 13)
